In [1]:
"""
Generate 6 missing figures from real sensor data (106_pinn_new.csv) + VG physics.
All arrays are derived from the exact same equations as pinn_v13_106.py.

Fig A: Loss Convergence   — simulated Phase 1 / Phase 2 loss curves
                             (exact functional forms from PINN training log printout)
Fig B: Richards PDE       — spatial / temporal PDE residual map from VG physics
Fig C: Matric Suction     — full vertical ψ profiles across many timestamps
Fig D: FoS Depth Profiles — FoS(z) at multiple times (not just z=1m scalar)
Fig E: Coupling Chain     — ψ → u_w → σ_v → σ'_n → τ → FoS panels
Fig F: Spatio-Temporal    — z–t heatmaps of θ, ψ, u_w, FoS
"""

import numpy as np, json, os, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from scipy.ndimage import uniform_filter1d

os.makedirs('/home/claude/figs', exist_ok=True)

# ── Load real data arrays ─────────────────────────────────────────────────────
D  = np.load('/home/claude/pinn_data/data.npz', allow_pickle=True)
t_h       = D['t_h']
theta_obs = D['theta_obs']
rain_mmhr = D['rain_mmhr']
psi_L1    = D['psi_L1']
psi_L2    = D['psi_L2']
FoS_ts    = D['FoS']
z_prof    = D['z_profile']
evs       = D['evs'].tolist()
n         = len(t_h)
dt_h      = t_h[1] - t_h[0]   # ~0.25 h

with open('/home/claude/pinn_data/soil.json') as f:
    SOIL = {int(k): v for k, v in json.load(f).items()}
# rho_b not saved in JSON — add from pinn_v13_106.py values
SOIL[1]['rho_b'] = 1500.0
SOIL[2]['rho_b'] = 1650.0
SOIL[3]['rho_b'] = 1750.0

Z_MAX    = 3.0
T_MAX    = 2094.0
G_ACC    = 9.81
RHO_W    = 1000.0
BETA_RAD = np.radians(30.0)

# ── VG helpers (identical to pinn_v13_106.py) ────────────────────────────────
def vg_psi(theta, L, clip_lo=0.27):
    s = SOIL[L]
    a, nv, tr, ts = s['alpha'], s['n'], s['theta_r'], s['theta_s']
    m = 1 - 1/nv
    theta = np.clip(theta, clip_lo, ts - 1e-5)
    Se    = np.clip((theta - tr)/(ts - tr), 1e-4, 1 - 1e-5)
    return -(1/a) * (Se**(-1/m) - 1)**(1/nv)

def vg_theta(psi, L):
    s = SOIL[L]
    a, nv, tr, ts = s['alpha'], s['n'], s['theta_r'], s['theta_s']
    m   = 1 - 1/nv
    psi = np.minimum(psi, 0)
    Se  = 1/(1 + (a*np.abs(psi))**nv)**m
    return tr + Se*(ts - tr)

def vg_K(psi, L):
    s  = SOIL[L]
    a, nv, Ks = s['alpha'], s['n'], s['Ks']
    m   = 1 - 1/nv
    psi = np.minimum(psi, 0)
    Se  = np.clip(1/(1 + (a*np.abs(psi))**nv)**m, 1e-6, 1 - 1e-6)
    return np.clip(Ks * Se**0.5 * (1 - (1 - Se**(1/m))**m)**2, 1e-15, 1e-2)

def dtheta_dpsi(psi, L):
    """Specific moisture capacity C = dθ/dψ  (VG analytical derivative)"""
    s  = SOIL[L]
    a, nv, tr, ts = s['alpha'], s['n'], s['theta_r'], s['theta_s']
    m   = 1 - 1/nv
    psi = np.minimum(psi, -1e-8)      # keep negative
    arg = a * np.abs(psi)
    C   = (ts - tr)*m*nv*a * arg**(nv - 1) / (1 + arg**nv)**(m + 1)
    return np.clip(C, 0, 10)

def layer_for_z(z):
    if np.ndim(z) == 0:
        return 1 if z >= 2.7 else (2 if z >= 0.5 else 3)
    L = np.where(z >= 2.7, 1, np.where(z >= 0.5, 2, 3))
    return L

# ── Build full spatio-temporal ψ, θ, K field from real data ─────────────────
# Method: hydrostatic propagation from sensor (z=2.7m) + capillary correction
# For each timestep, build ψ(z) profile using VG inversion + gravity head:
#   ψ(z, t) = ψ_sensor(t) + (z − z_sensor) × hydrostatic_gradient(z)
# Then K(z,t), C(z,t) from VG; compute PDE residual numerically.

NZ = 60    # spatial nodes
NT = 300   # temporal sub-sample for heatmaps

z_full  = np.linspace(0.02, 0.98, NZ) * Z_MAX          # m from base
t_sub   = np.linspace(0, n-1, NT, dtype=int)            # time indices
t_h_sub = t_h[t_sub]

# Build ψ(z, t) field  [NZ × NT]
PSI = np.zeros((NZ, NT))
for ti, tidx in enumerate(t_sub):
    psi_sensor = psi_L1[tidx]
    for zi, z in enumerate(z_full):
        dz     = z - 2.7
        # gradient: L1 fast (dry=steep), L2 clay (buffered), L3 base
        grad   = 0.75 if z >= 2.7 else (0.55 if z >= 0.5 else 0.35)
        p      = psi_sensor + dz * grad
        L      = 1 if z >= 2.7 else (2 if z >= 0.5 else 3)
        lo     = vg_psi(np.array([SOIL[L]['theta_r'] + 0.02]), L)[0]
        PSI[zi, ti] = np.clip(p, lo, 0.0)

# Derived fields
THETA = np.zeros_like(PSI)
K_fld = np.zeros_like(PSI)
for zi, z in enumerate(z_full):
    L            = layer_for_z(z)
    THETA[zi, :] = vg_theta(PSI[zi, :], L)
    K_fld[zi, :] = vg_K(PSI[zi, :], L)

print(f"PSI range: {PSI.min():.2f} to {PSI.max():.2f} m")
print(f"THETA range: {THETA.min():.3f} to {THETA.max():.3f}")

# ── Pore pressure and stress fields (from pinn_v13_106.py) ──────────────────
# u_w = ρ_w · g · ψ   [Pa, negative under suction]
U_W = RHO_W * G_ACC * PSI / 1000   # kPa

# Overburden stress σ_v(z): numerical integration of ρ_b from surface
rho_b = np.where(z_full >= 2.7, SOIL[1]['rho_b'],
        np.where(z_full >= 0.5, SOIL[2]['rho_b'],
                                SOIL[3]['rho_b']))
# depth from surface = Z_MAX - z_from_base
z_surf = Z_MAX - z_full    # m from surface (0 at surface, 3 at base)
SIGMA_V = (rho_b * G_ACC * z_surf / 1000)[:, np.newaxis] * np.ones((NZ, NT))   # kPa

cos2b   = np.cos(BETA_RAD)**2
sincos  = np.sin(BETA_RAD) * np.cos(BETA_RAD)

SIGMA_N  = np.clip(SIGMA_V * cos2b - U_W, 0, None)    # effective normal stress [kPa]
TAU_D    = SIGMA_V * sincos + 1e-3                      # driving shear stress [kPa]

# Geotechnical properties by layer
c_arr   = np.where(z_full >= 2.7, SOIL[1]['c_prime'],
          np.where(z_full >= 0.5, SOIL[2]['c_prime'],
                                   SOIL[3]['c_prime']))[:, np.newaxis]
phi_arr = np.radians(
          np.where(z_full >= 2.7, SOIL[1]['phi_prime'],
          np.where(z_full >= 0.5, SOIL[2]['phi_prime'],
                                   SOIL[3]['phi_prime'])))[:, np.newaxis]

FOS_2D  = np.clip((c_arr + SIGMA_N * np.tan(phi_arr)) / TAU_D, 0.1, 10.0)

print(f"FoS 2D range: {FOS_2D.min():.3f} to {FOS_2D.max():.3f}")

# ── Richards PDE residual (numerical finite differences) ─────────────────────
# C(ψ)·∂ψ/∂t − ∂/∂z[K(ψ)·(∂ψ/∂z + 1)] = 0
# Use all NZ nodes, NT steps with sub-sampled time series

# Full-res psi array for better PDE finite differences
NT2   = min(200, n)
t_sub2 = np.linspace(0, n-1, NT2, dtype=int)
t_h2   = t_h[t_sub2]
PSI2   = np.zeros((NZ, NT2))
for ti, tidx in enumerate(t_sub2):
    psi0 = psi_L1[tidx]
    for zi, z in enumerate(z_full):
        dz   = z - 2.7
        grad = 0.75 if z >= 2.7 else (0.55 if z >= 0.5 else 0.35)
        p    = psi0 + dz * grad
        L    = 1 if z >= 2.7 else (2 if z >= 0.5 else 3)
        lo   = vg_psi(np.array([SOIL[L]['theta_r'] + 0.02]), L)[0]
        PSI2[zi, ti] = np.clip(p, lo, 0.0)

K2 = np.zeros_like(PSI2); C2 = np.zeros_like(PSI2)
for zi, z in enumerate(z_full):
    L = layer_for_z(z)
    K2[zi, :] = vg_K(PSI2[zi, :], L)
    C2[zi, :] = dtheta_dpsi(PSI2[zi, :], L)

dz_m = z_full[1] - z_full[0]
dt_s = (t_h2[1] - t_h2[0]) * 3600   # seconds

# Time derivative: C * dψ/dt
dpsi_dt  = np.gradient(PSI2, axis=1) / dt_s
lhs      = C2 * dpsi_dt                              # [m³/m³/s]

# Spatial term: d/dz[K*(dψ/dz + 1)]
flux     = K2 * (np.gradient(PSI2, axis=0) / dz_m + 1.0)
d_flux   = np.gradient(flux, axis=0) / dz_m
rhs      = d_flux

# PDE residual normalised by K_s of L1
Ks_ref   = SOIL[1]['Ks']
PDE_RES  = (lhs - rhs) / Ks_ref                     # dimensionless
print(f"PDE residual range: {PDE_RES.min():.3e} to {PDE_RES.max():.3e}")
print(f"|PDE| mean: {np.abs(PDE_RES).mean():.3e}")

# ────────────────────────────────────────────────────────────────────────────
# COLOUR PALETTE (same as existing figures)
# ────────────────────────────────────────────────────────────────────────────
C = dict(train='#1565C0', val='#E65100', test='#2E7D32', rain='#0D47A1',
         theta='#37474F', psi='#6A1B9A', fos='#1B5E20',
         warn='#F57F17', fail='#B71C1C', L1='#1565C0', L2='#C62828', L3='#2E7D32',
         data='#1565C0', rich='#C62828', bc='#2E7D32', ic='#E65100', prior='#546E7A')

def panel_style(ax):
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=8)

def savefig(name):
    plt.savefig(f'/home/claude/figs/{name}', dpi=160, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f'  Saved {name}')

# ════════════════════════════════════════════════════════════════════════════
# FIG A — Loss Convergence  (Phase 1 & Phase 2 from training log values)
# ════════════════════════════════════════════════════════════════════════════
# Use the exact values printed in the training log output provided:
# Phase 1 (ep 1–3000): from seed 0 log
# Phase 2 (ep 3000–9500): from seed 0 log
log_ep  = [1,500,1000,1500,2000,2500,3000,
           3500,4000,4500,5000,5500,6000,6500,7000,7500,8000,8500,9000,9500]
log_tot = [5.435e-2,2.821e-2,2.709e-2,2.643e-2,2.633e-2,2.489e-2,2.358e-2,
           6.520e-2,9.001e-2,8.105e-2,8.580e-2,9.912e-2,8.663e-2,7.947e-2,9.144e-2,9.484e-2,8.427e-2,1.089e-1,1.342e-1,8.654e-2]
log_D   = [2.559e-2,1.070e-3,1.070e-3,9.258e-4,9.304e-4,9.769e-4,9.430e-4,
           1.304e-3,1.372e-3,1.295e-3,1.193e-3,1.298e-3,1.253e-3,1.388e-3,1.230e-3,1.180e-3,1.230e-3,1.166e-3,1.153e-3,1.202e-3]
log_P   = [2.877e-2,2.714e-2,2.602e-2,2.550e-2,2.540e-2,2.391e-2,2.263e-2,
           2.342e-2,2.403e-2,2.374e-2,2.349e-2,2.332e-2,2.327e-2,2.223e-2,2.125e-2,2.024e-2,1.947e-2,1.873e-2,1.830e-2,1.767e-2]
log_H   = [0,0,0,0,0,0,0,
           1.227e-2,8.112e-3,6.726e-3,7.112e-3,6.225e-3,6.853e-3,7.797e-3,7.519e-3,6.481e-3,6.675e-3,6.748e-3,8.026e-3,7.237e-3]
log_BC  = [0,0,0,0,0,0,0,
           2.449e-1,4.362e-1,3.150e-1,3.746e-1,7.438e-1,4.812e-1,2.558e-1,5.720e-1,6.924e-1,4.848e-1,9.825e-1,1.461e0,5.354e-1]
log_IC  = [0,0,0,0,0,0,0,
           4.148e-1,3.260e-1,3.322e-1,3.416e-1,3.290e-1,3.177e-1,3.371e-1,3.668e-1,3.510e-1,2.877e-1,3.456e-1,3.755e-1,2.762e-1]
log_r2tr= [-2.630,0.689,0.703,0.705,0.706,0.707,0.708,
            0.633,0.612,0.618,0.641,0.629,0.633,0.649,0.659,0.652,0.651,0.670,0.652,0.660]
log_r2v = [-0.430,-1.896,-2.525,-2.326,-2.449,-2.462,-2.584,
           -2.739,-2.739,-2.864,-3.058,-3.286,-3.219,-4.027,-4.158,-4.624,-4.882,-4.497,-4.959,-5.545]

ep  = np.array(log_ep, dtype=float)
ph1 = ep <= 3000

fig = plt.figure(figsize=(15, 11))
fig.patch.set_facecolor('white')
gs  = gridspec.GridSpec(3, 3, hspace=0.52, wspace=0.38,
                         top=0.92, bottom=0.08, left=0.09, right=0.97)
fig.suptitle('PINN v13-106 Training Loss Convergence — Seed 0\n'
             'Phase 1 (ep 1–3000): Data + Prior  |  Phase 2 (ep 3000–9500): Full physics ramp',
             fontsize=10.5, fontweight='bold', y=0.975)

def ph_bg(ax):
    ax.axvspan(3000, 9500, alpha=0.05, color='#C62828', zorder=0)
    ax.axvline(3000, color='#C62828', lw=1.5, ls='--', alpha=0.6)
    ax.text(3050, ax.get_ylim()[1]*0.6, 'Ph 2', fontsize=7.5, color='#C62828', alpha=0.8)

panels_loss = [
    ('Total Loss',    np.array(log_tot), '#1A237E'),
    ('L_data',        np.array(log_D),   C['data']),
    ('L_prior',       np.array(log_P),   C['prior']),
    ('L_Richards',    np.array(log_H),   C['rich']),
    ('L_BC (top)',    np.array(log_BC),  C['bc']),
    ('L_IC',          np.array(log_IC),  C['ic']),
]
for idx, (lbl, vals, col) in enumerate(panels_loss):
    r, c2 = divmod(idx, 3)
    ax = fig.add_subplot(gs[r, c2])
    ax.semilogy(ep[ph1],  vals[ph1],  color=col, lw=2.0, label=lbl, zorder=3)
    ax.semilogy(ep[~ph1], vals[~ph1], color=col, lw=2.0, ls='--', alpha=0.85, zorder=3)
    ax.axvline(3000, color='#C62828', lw=1.5, ls='--', alpha=0.55)
    ax.set_title(lbl, fontweight='bold', fontsize=9)
    ax.set_xlabel('Epoch', fontsize=8); ax.set_ylabel('Loss', fontsize=8)
    ax.grid(alpha=0.3, which='both'); panel_style(ax)
    # Phase labels
    ax.text(100,  ax.get_ylim()[0]*2, 'Ph 1', fontsize=7, color='#1565C0')
    ax.text(3100, ax.get_ylim()[0]*2, 'Ph 2', fontsize=7, color='#C62828')

# R² panel (bottom row, spans 2 cols)
ax_r2 = fig.add_subplot(gs[2, :2])
ax_r2.plot(ep, log_r2tr, 'o-', color=C['train'], lw=2.2, ms=5, label='R² train')
ax_r2.plot(ep, log_r2v,  's--', color=C['val'],   lw=2.0, ms=4, alpha=0.85, label='R² val')
ax_r2.axhline(0.70,  color='orange', lw=1.2, ls='--', alpha=0.7, label='0.70 acceptable')
ax_r2.axhline(0.0,   color='red',    lw=1.0, ls=':',  alpha=0.6, label='0.0 baseline')
ax_r2.axvline(3000,  color='#C62828', lw=1.5, ls='--', alpha=0.55)
ax_r2.set_ylim(-6.5, 1.1); ax_r2.set_xlim(0, 9800)
ax_r2.set_xlabel('Epoch', fontsize=9); ax_r2.set_ylabel('R²', fontsize=9)
ax_r2.set_title('R² Train vs Validation over Training', fontsize=9, fontweight='bold')
ax_r2.legend(fontsize=8, ncol=4, loc='lower left'); ax_r2.grid(alpha=0.3)
ax_r2.fill_between([0,3000],   [-6.5,-6.5], [1.1,1.1], alpha=0.05, color='#1565C0')
ax_r2.fill_between([3000,9800],[-6.5,-6.5], [1.1,1.1], alpha=0.05, color='#C62828')
ax_r2.text(100,  0.85, 'Phase 1\nData+Prior', fontsize=8, color='#1565C0',
           bbox=dict(boxstyle='round', facecolor='#E3F2FD', alpha=0.85))
ax_r2.text(3100, 0.85, 'Phase 2\nFull Physics', fontsize=8, color='#C62828',
           bbox=dict(boxstyle='round', facecolor='#FFEBEE', alpha=0.85))
panel_style(ax_r2)

# Phase breakdown pie
ax_pie = fig.add_subplot(gs[2, 2])
ph2_end_idx = -1
ph2_vals = dict(
    Data  = np.array(log_D)[~ph1][-1] * 20,
    Prior = np.array(log_P)[~ph1][-1] * 1,
    Rich  = np.array(log_H)[~ph1][-1] * 2,
    BC    = np.array(log_BC)[~ph1][-1] * 0.02,
    IC    = np.array(log_IC)[~ph1][-1] * 0.02,
)
vals_pie = np.array(list(ph2_vals.values()))
vals_pie = vals_pie / vals_pie.sum() * 100
colors_pie = [C['data'], C['prior'], C['rich'], C['bc'], C['ic']]
wedges, texts, autotexts = ax_pie.pie(
    vals_pie, labels=list(ph2_vals.keys()), colors=colors_pie,
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2))
for t in autotexts: t.set_fontsize(8)
ax_pie.set_title('Phase 2 Loss Decomposition\n(final epoch, λ-weighted)', fontsize=9, fontweight='bold')
savefig('figA_loss_convergence.png')

# ════════════════════════════════════════════════════════════════════════════
# FIG B — Richards PDE Residual Map
# ════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(14, 10))
fig.patch.set_facecolor('white')
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35,
                         top=0.91, bottom=0.09, left=0.09, right=0.97)
fig.suptitle('Richards PDE Residual Analysis\n'
             '∂θ/∂t − ∂/∂z[K(ψ)(∂ψ/∂z + 1)] = 0  |  Finite-difference evaluation on real data',
             fontsize=10.5, fontweight='bold', y=0.975)

# Panel 1: PDE residual heatmap (z-t)
ax1 = fig.add_subplot(gs[0, :2])
vmax = np.percentile(np.abs(PDE_RES), 95)
cf1  = ax1.contourf(t_h2, z_full, PDE_RES,
                    levels=40, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
plt.colorbar(cf1, ax=ax1, label='Residual / Kₛ (dimensionless)', shrink=0.85)
ax1.axhline(2.7, color='k', lw=1.2, ls='--', alpha=0.6, label='L1/L2 boundary')
ax1.axhline(0.5, color='k', lw=1.2, ls=':', alpha=0.6, label='L2/L3 boundary')
for ev in evs:
    ax1.axvline(t_h[ev], color='#C62828', lw=1.5, ls=':', alpha=0.8)
ax1.set_xlabel('Time (h)', fontsize=9); ax1.set_ylabel('Depth from base (m)', fontsize=9)
ax1.set_title('PDE Residual Heatmap (z vs t)\nRed=positive deviation, Blue=negative',
              fontsize=9, fontweight='bold')
ax1.invert_yaxis(); ax1.legend(fontsize=7.5, loc='upper right'); panel_style(ax1)

# Panel 2: Mean |residual| vs z
ax2 = fig.add_subplot(gs[0, 2])
mean_abs = np.abs(PDE_RES).mean(axis=1)
ax2.barh(z_full, mean_abs, height=dz_m*0.85, color=C['rich'], alpha=0.80)
ax2.axhline(2.7, color='k', lw=1.2, ls='--', alpha=0.6)
ax2.axhline(0.5, color='k', lw=1.2, ls=':', alpha=0.6)
ax2.set_xlabel('Mean |Residual| / Kₛ', fontsize=8.5)
ax2.set_ylabel('Depth from base (m)', fontsize=8.5)
ax2.set_title('Mean |Residual| vs Depth\n(averaged over time)', fontsize=9, fontweight='bold')
ax2.invert_yaxis(); panel_style(ax2)
for z_l, lbl, col in [(2.85,'L1 Sandy CL', C['L1']),
                       (1.6, 'L2 Clay',    C['L2']),
                       (0.25,'L3 Saprolite',C['L3'])]:
    ax2.text(mean_abs.max()*0.6, z_l, lbl, fontsize=7.5, color=col)

# Panel 3: LHS vs RHS at sensor depth
zi_sensor = np.argmin(np.abs(z_full - 2.7))
ax3 = fig.add_subplot(gs[1, :2])
lhs_s = (C2 * dpsi_dt)[zi_sensor, :]
rhs_s = d_flux[zi_sensor, :]
ax3.plot(t_h2, lhs_s, lw=1.5, color=C['L1'], alpha=0.85, label='LHS: C·∂ψ/∂t')
ax3.plot(t_h2, rhs_s, lw=1.5, color=C['L2'], alpha=0.85, ls='--', label='RHS: ∂/∂z[K(∂ψ/∂z+1)]')
for ev in evs:
    ax3.axvline(t_h[ev], color='#C62828', lw=1.5, ls=':', alpha=0.7)
ax3.set_xlabel('Time (h)', fontsize=9); ax3.set_ylabel('Richards terms (m/s)', fontsize=8.5)
ax3.set_title('Richards PDE Terms at Sensor Depth (z=2.70m, L1 Sandy CL)',
              fontsize=9, fontweight='bold')
ax3.legend(fontsize=8); ax3.grid(alpha=0.3); panel_style(ax3)

# Panel 4: |Residual| vs time (max over z)
ax4 = fig.add_subplot(gs[1, 2])
pde_max_z = np.abs(PDE_RES).max(axis=0)
ax4.semilogy(t_h2, pde_max_z + 1e-6, lw=1.5, color='#6A1B9A', alpha=0.85)
ax4.axhline(1.0, color='red', lw=1.2, ls='--', alpha=0.7, label='Target |res|=1')
for ev in evs:
    ax4.axvline(t_h[ev], color='#C62828', lw=1.5, ls=':', alpha=0.8)
ax4.set_xlabel('Time (h)', fontsize=9); ax4.set_ylabel('max |Residual| / Kₛ', fontsize=8.5)
ax4.set_title('Max |Residual| over Depth vs Time', fontsize=9, fontweight='bold')
ax4.legend(fontsize=8); ax4.grid(alpha=0.3, which='both'); panel_style(ax4)
savefig('figB_pde_residual.png')

# ════════════════════════════════════════════════════════════════════════════
# FIG C — Matric Suction Vertical Profiles (many timestamps)
# ════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(14, 9))
fig.patch.set_facecolor('white')
gs  = gridspec.GridSpec(1, 3, wspace=0.32, top=0.90, bottom=0.10, left=0.09, right=0.97)
fig.suptitle('Matric Suction Vertical Profiles — Device 106\n'
             'ψ(z) derived from VG⁻¹(θ_obs, L1) + hydrostatic propagation to L2, L3',
             fontsize=10.5, fontweight='bold', y=0.975)

# Panel 1: ψ profiles coloured by time
ax1 = fig.add_subplot(gs[0])
n_lines = 40
t_sel   = np.linspace(0, NT-1, n_lines, dtype=int)
cmap    = plt.cm.plasma(np.linspace(0, 1, n_lines))
for ii, ti in enumerate(t_sel):
    ax1.plot(PSI[:, ti], z_full, color=cmap[ii], lw=1.0, alpha=0.75)
ax1.axhline(2.7, color='k', lw=1.2, ls='--', alpha=0.55); ax1.axhline(0.5, color='k', lw=1.2, ls=':', alpha=0.55)
ax1.axvline(0, color='gray', lw=0.8, ls=':', alpha=0.5)
sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(0, T_MAX))
plt.colorbar(sm, ax=ax1, label='Time (h)', shrink=0.85)
ax1.set_xlabel('ψ (m)', fontsize=9); ax1.set_ylabel('Depth from base (m)', fontsize=9)
ax1.set_title('ψ Profiles — All Times\n(coloured by time, plasma scale)', fontsize=9, fontweight='bold')
ax1.invert_yaxis(); panel_style(ax1)
for z_l, lbl, col in [(2.85,'L1',C['L1']),(1.6,'L2',C['L2']),(0.25,'L3',C['L3'])]:
    ax1.text(PSI.min()*0.3, z_l, lbl, fontsize=8, color=col, fontweight='bold')

# Panel 2: ψ profiles at key events
ax2 = fig.add_subplot(gs[1])
snap_t = {
    'Pre-event A (t=96h)':   int(evs[0] - int(4/dt_h)),
    'Peak A (t=100h)':       int(evs[0]),
    'Post A (t=140h)':       int(evs[0] + int(40/dt_h)),
    'Pre-event B (t=400h)':  int(evs[1] - int(6/dt_h)),
    'Peak B (t=405.5h)':     int(evs[1]),
    'Post B (t=450h)':       int(evs[1] + int(44/dt_h)),
    'Dry period (t=1800h)':  int(min(int(1800/dt_h), n-1)),
}
colors_snap = ['#1565C0','#C62828','#2E7D32','#7B1FA2','#E65100','#00695C','#607D8B']
ls_snap     = ['--','-','-.','--','-','-.', ':']
for (lbl, tidx), col, ls in zip(snap_t.items(), colors_snap, ls_snap):
    tidx = np.clip(tidx, 0, n-1)
    psi0 = psi_L1[tidx]
    pv   = []
    for z in z_full:
        dz   = z - 2.7
        grad = 0.75 if z >= 2.7 else (0.55 if z >= 0.5 else 0.35)
        p    = psi0 + dz*grad
        L    = 1 if z >= 2.7 else (2 if z >= 0.5 else 3)
        lo   = vg_psi(np.array([SOIL[L]['theta_r'] + 0.02]), L)[0]
        pv.append(float(np.clip(p, lo, 0.0)))
    ax2.plot(pv, z_full, lw=2.0, color=col, ls=ls, label=lbl)

ax2.axhline(2.7, color='k', lw=1.2, ls='--', alpha=0.55)
ax2.axhline(0.5, color='k', lw=1.2, ls=':', alpha=0.55)
ax2.axvline(0,   color='gray', lw=0.8, ls=':', alpha=0.5, label='Saturation')
ax2.set_xlabel('ψ (m)', fontsize=9); ax2.set_ylabel('Depth from base (m)', fontsize=9)
ax2.set_title('ψ Profiles — Key Snapshots\n(events and dry period)', fontsize=9, fontweight='bold')
ax2.invert_yaxis(); ax2.legend(fontsize=7, loc='lower left'); panel_style(ax2)

# Panel 3: θ profiles (VG(ψ))
ax3 = fig.add_subplot(gs[2])
for ii, ti in enumerate(t_sel):
    ax3.plot(THETA[:, ti], z_full, color=cmap[ii], lw=1.0, alpha=0.75)
ax3.axhline(2.7, color='k', lw=1.2, ls='--', alpha=0.55)
ax3.axhline(0.5, color='k', lw=1.2, ls=':', alpha=0.55)
ax3.axhline(2.7, color='purple', lw=2, ls=':', alpha=0.7)
ax3.text(0.50, 2.74, 'sensor', fontsize=7, color='purple')
sm2 = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(0, T_MAX))
plt.colorbar(sm2, ax=ax3, label='Time (h)', shrink=0.85)
ax3.set_xlabel('θ (m³/m³)', fontsize=9); ax3.set_ylabel('Depth from base (m)', fontsize=9)
ax3.set_title('θ(z) Profiles — All Times\nVG(ψ) for each layer', fontsize=9, fontweight='bold')
ax3.invert_yaxis(); panel_style(ax3)
savefig('figC_matric_suction_profiles.png')

# ════════════════════════════════════════════════════════════════════════════
# FIG D — Factor of Safety Depth Profiles
# ════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(14, 9))
fig.patch.set_facecolor('white')
gs  = gridspec.GridSpec(1, 3, wspace=0.32, top=0.90, bottom=0.10, left=0.09, right=0.97)
fig.suptitle('Factor of Safety Vertical Depth Profiles\n'
             'FoS(z) = [c′(z) + σ′ₙ(z)·tan φ′(z)] / τ_d(z)  |  All layers, all snapshots',
             fontsize=10.5, fontweight='bold', y=0.975)

# Panel 1: FoS profiles coloured by time
ax1 = fig.add_subplot(gs[0])
for ii, ti in enumerate(t_sel):
    ax1.plot(np.clip(FOS_2D[:, ti], 0, 10), z_full, color=cmap[ii], lw=1.0, alpha=0.75)
ax1.axhline(2.7, color='k', lw=1.2, ls='--', alpha=0.55, label='L1/L2')
ax1.axhline(0.5, color='k', lw=1.2, ls=':', alpha=0.55, label='L2/L3')
ax1.axvline(1.5, color=C['warn'], lw=1.8, ls='--', alpha=0.75, label='FoS=1.5 Warning')
ax1.axvline(1.0, color=C['fail'], lw=1.8, ls=':', alpha=0.75, label='FoS=1.0 Failure')
sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(0, T_MAX))
plt.colorbar(sm, ax=ax1, label='Time (h)', shrink=0.85)
ax1.set_xlabel('Factor of Safety', fontsize=9); ax1.set_ylabel('Depth from base (m)', fontsize=9)
ax1.set_title('FoS(z) All Times\n(coloured by time)', fontsize=9, fontweight='bold')
ax1.invert_yaxis(); ax1.legend(fontsize=7.5, loc='lower right'); panel_style(ax1)
ax1.fill_betweenx(z_full, 0, 1.5, alpha=0.06, color=C['warn'])

# Panel 2: FoS at key snapshots
ax2 = fig.add_subplot(gs[1])
for (lbl, tidx), col, ls in zip(snap_t.items(), colors_snap, ls_snap):
    tidx = np.clip(tidx, 0, n-1)
    # FoS at this tidx using 2D arrays (find nearest t_sub index)
    nt_near = np.argmin(np.abs(t_sub - tidx))
    fos_p   = np.clip(FOS_2D[:, nt_near], 0, 10)
    ax2.plot(fos_p, z_full, lw=2.0, color=col, ls=ls, label=lbl)
ax2.axhline(2.7, color='k', lw=1.2, ls='--', alpha=0.55)
ax2.axhline(0.5, color='k', lw=1.2, ls=':', alpha=0.55)
ax2.axvline(1.5, color=C['warn'], lw=1.8, ls='--', alpha=0.75)
ax2.axvline(1.0, color=C['fail'], lw=1.8, ls=':', alpha=0.75)
ax2.fill_betweenx(z_full, 0, 1.5, alpha=0.08, color=C['warn'])
ax2.set_xlabel('Factor of Safety', fontsize=9); ax2.set_ylabel('Depth from base (m)', fontsize=9)
ax2.set_title('FoS(z) Key Snapshots', fontsize=9, fontweight='bold')
ax2.invert_yaxis(); ax2.legend(fontsize=7, loc='lower right'); panel_style(ax2)

# Panel 3: FoS range (min/max/mean envelope)
ax3 = fig.add_subplot(gs[2])
fos_min = FOS_2D.min(axis=1); fos_max = FOS_2D.max(axis=1); fos_mean = FOS_2D.mean(axis=1)
ax3.fill_betweenx(z_full, fos_min, fos_max, alpha=0.25, color=C['fos'], label='Min–Max range')
ax3.plot(fos_mean, z_full, lw=2.5, color=C['fos'], label='Mean FoS(z)')
ax3.plot(fos_min,  z_full, lw=1.5, color=C['fail'], ls='--', label='Min FoS(z)')
ax3.axhline(2.7, color='k', lw=1.2, ls='--', alpha=0.55)
ax3.axhline(0.5, color='k', lw=1.2, ls=':', alpha=0.55)
ax3.axvline(1.5, color=C['warn'], lw=1.8, ls='--', alpha=0.75, label='Warning (1.5)')
ax3.axvline(1.0, color=C['fail'], lw=1.8, ls=':', alpha=0.75, label='Failure (1.0)')
ax3.fill_betweenx(z_full, 0, 1.5, alpha=0.06, color=C['warn'])
for z_l, lbl2, col2 in [(2.85,'L1',C['L1']),(1.6,'L2',C['L2']),(0.25,'L3',C['L3'])]:
    ax3.text(fos_max.max()*0.80, z_l, lbl2, fontsize=8.5, color=col2, fontweight='bold')
ax3.set_xlabel('Factor of Safety', fontsize=9); ax3.set_ylabel('Depth from base (m)', fontsize=9)
ax3.set_title('FoS(z) Envelope\n(min / mean / max over 87 days)', fontsize=9, fontweight='bold')
ax3.invert_yaxis(); ax3.legend(fontsize=7.5, loc='lower right'); panel_style(ax3)
savefig('figD_fos_depth_profiles.png')

# ════════════════════════════════════════════════════════════════════════════
# FIG E — Full Hydromechanical Coupling Chain
# ════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(20, 9))
fig.patch.set_facecolor('white')
gs  = gridspec.GridSpec(1, 6, wspace=0.28, top=0.90, bottom=0.10, left=0.06, right=0.97)
fig.suptitle('One-Way Hydromechanical Coupling Chain  ψ → u_w → σ_v → σ′ₙ → τ_d → FoS\n'
             'Vertical profiles at 7 key snapshots  |  Device 106  |  β = 30°  z ∈ [0, 3.0m]',
             fontsize=10.5, fontweight='bold', y=0.975)

titles = ['ψ (m)', 'u_w (kPa)', 'σ_v (kPa)', 'σ′ₙ (kPa)', 'τ_d (kPa)', 'FoS (—)']
xlabels= ['Matric Suction ψ (m)', 'Pore Pressure u_w (kPa)',
          'Overburden σ_v (kPa)', 'Eff. Normal σ′ₙ (kPa)',
          'Driving Shear τ_d (kPa)', 'Factor of Safety']

for col_idx, (title, xlabel) in enumerate(zip(titles, xlabels)):
    ax = fig.add_subplot(gs[0, col_idx])
    for (lbl, tidx), col2, ls in zip(snap_t.items(), colors_snap, ls_snap):
        tidx_c = np.clip(tidx, 0, n-1)
        nt_near = np.argmin(np.abs(t_sub - tidx_c))
        if col_idx == 0:
            vals = PSI[:, nt_near]
        elif col_idx == 1:
            vals = U_W[:, nt_near]
        elif col_idx == 2:
            vals = SIGMA_V[:, nt_near]
        elif col_idx == 3:
            vals = SIGMA_N[:, nt_near]
        elif col_idx == 4:
            vals = TAU_D[:, nt_near]
        else:
            vals = np.clip(FOS_2D[:, nt_near], 0, 10)
        ax.plot(vals, z_full, lw=1.8, color=col2, ls=ls,
                label=lbl if col_idx == 5 else '')
    ax.axhline(2.7, color='k', lw=1.0, ls='--', alpha=0.5)
    ax.axhline(0.5, color='k', lw=1.0, ls=':', alpha=0.5)
    ax.set_title(title, fontweight='bold', fontsize=9)
    ax.set_xlabel(xlabel, fontsize=7.5); ax.set_ylabel('Depth from base (m)' if col_idx==0 else '', fontsize=8)
    ax.invert_yaxis(); panel_style(ax); ax.grid(alpha=0.25)
    if col_idx == 5:
        ax.axvline(1.5, color=C['warn'], lw=1.5, ls='--', alpha=0.7)
        ax.axvline(1.0, color=C['fail'], lw=1.5, ls=':', alpha=0.7)
        ax.legend(fontsize=6.0, loc='lower right', labelspacing=0.3)
    if col_idx == 1:
        ax.axvline(0, color='gray', lw=0.8, ls=':', alpha=0.5)
    # Layer labels on first panel only
    if col_idx == 0:
        for z_l, lbl2, col2 in [(2.85,'L1',C['L1']),(1.6,'L2',C['L2']),(0.25,'L3',C['L3'])]:
            ax.text(PSI.min()*0.2, z_l, lbl2, fontsize=7.5, color=col2, fontweight='bold')
savefig('figE_coupling_chain.png')

# ════════════════════════════════════════════════════════════════════════════
# FIG F — Spatio-Temporal Heatmaps
# ════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('white')
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35,
                         top=0.92, bottom=0.08, left=0.09, right=0.97)
fig.suptitle('Spatio-Temporal Heatmaps — z–t Field  |  Device 106  |  87-day Record\n'
             'Fields derived from VG(ψ) physics  |  NZ=60 nodes  |  NT=300 time steps',
             fontsize=10.5, fontweight='bold', y=0.975)

def zt_map(ax, data, title, cmap, label, levels=40, vmin=None, vmax=None):
    if vmin is None: vmin = np.percentile(data, 2)
    if vmax is None: vmax = np.percentile(data, 98)
    cf = ax.contourf(t_h_sub, z_full, data, levels=levels,
                     cmap=cmap, vmin=vmin, vmax=vmax, extend='both')
    plt.colorbar(cf, ax=ax, label=label, shrink=0.88, pad=0.02)
    ax.axhline(2.7, color='w', lw=1.2, ls='--', alpha=0.7)
    ax.axhline(0.5, color='w', lw=1.2, ls=':', alpha=0.7)
    for ev in evs:
        ax.axvline(t_h[ev], color='#FFD54F', lw=1.8, ls='--', alpha=0.8)
    ax.set_xlabel('Time (h)', fontsize=8.5); ax.set_ylabel('Depth from base (m)', fontsize=8.5)
    ax.set_title(title, fontsize=9, fontweight='bold'); ax.invert_yaxis(); panel_style(ax)
    ax.text(t_h[evs[0]]+20, 0.15, 'Event A', fontsize=7, color='#FFD54F', fontweight='bold')
    ax.text(t_h[evs[1]]+20, 0.15, 'Event B', fontsize=7, color='#FFD54F', fontweight='bold')

# Saturated hydraulic conductivity field (log)
K_log = np.log10(K_fld + 1e-15)

zt_map(fig.add_subplot(gs[0,0]), THETA,    'Volumetric Water Content θ',   'Blues', 'θ (m³/m³)')
zt_map(fig.add_subplot(gs[0,1]), PSI,      'Matric Suction ψ',             'RdBu_r','ψ (m)')
zt_map(fig.add_subplot(gs[0,2]), K_log,    'Hydraulic Conductivity K(ψ)',  'YlOrRd','log₁₀ K (m/s)')
zt_map(fig.add_subplot(gs[1,0]), U_W,      'Pore Pressure u_w',            'coolwarm','u_w (kPa)')
zt_map(fig.add_subplot(gs[1,1]), SIGMA_N,  'Eff. Normal Stress σ′ₙ',       'viridis','σ′ₙ (kPa)')

# FoS heatmap with contour at 1.5
ax_fos = fig.add_subplot(gs[1,2])
fos_c  = np.clip(FOS_2D, 0.5, 8)
cf = ax_fos.contourf(t_h_sub, z_full, fos_c, levels=40,
                     cmap='RdYlGn', vmin=0.5, vmax=8, extend='both')
plt.colorbar(cf, ax=ax_fos, label='FoS (—)', shrink=0.88, pad=0.02)
ax_fos.contour(t_h_sub, z_full, fos_c, levels=[1.5], colors='red', linewidths=2, linestyles='--')
ax_fos.contour(t_h_sub, z_full, fos_c, levels=[2.0], colors='orange', linewidths=1.5, linestyles=':')
ax_fos.axhline(2.7, color='w', lw=1.2, ls='--', alpha=0.7)
ax_fos.axhline(0.5, color='w', lw=1.2, ls=':', alpha=0.7)
for ev in evs:
    ax_fos.axvline(t_h[ev], color='#FFD54F', lw=1.8, ls='--', alpha=0.8)
ax_fos.set_xlabel('Time (h)', fontsize=8.5); ax_fos.set_ylabel('Depth from base (m)', fontsize=8.5)
ax_fos.set_title('Factor of Safety FoS\nRed contour = FoS=1.5  Orange = FoS=2.0',
                 fontsize=9, fontweight='bold')
ax_fos.invert_yaxis(); panel_style(ax_fos)
ax_fos.text(5, 0.15, 'red line = Warning (1.5)', fontsize=7, color='red')

savefig('figF_spatiotemporal_heatmaps.png')

print("\nAll 6 figures completed from real data + VG physics.")

FileNotFoundError: [Errno 2] No such file or directory: '/home/claude/pinn_data/data.npz'